In [1]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

FILE = "Water Quality.csv"  # Change to full path if needed
OUT = "task1_categorical_outputs"
os.makedirs(OUT, exist_ok=True)

if not Path(FILE).exists():
    raise FileNotFoundError(f"Cannot find {FILE!r}; set FILE to the full CSV path.")
df = pd.read_csv(FILE, low_memory=False)
required = {"CharacteristicName", "ResultValue"}
if required - set(df.columns):
    raise ValueError(f"Missing columns: {required - set(df.columns)}")
df["ResultValue"] = pd.to_numeric(df["ResultValue"], errors="coerce")
df = df.dropna(subset=["CharacteristicName", "ResultValue"])
candidates = ["ActivityMediaName", "MonitoringLocationType", "ActivityType", "ResultUnit",
              "ResultSampleFraction", "ResultStatusID", "MethodSpeciation", "MonitoringLocationID"]
sns.set_theme(style="whitegrid")
summary = []
for col in [c for c in candidates if c in df and df[c].nunique(dropna=True) > 1]:
    df[col] = df[col].fillna("Missing").astype(str)
    top = df[col].value_counts().head(15).index
    sub = df[df[col].isin(top)]
    counts = sub[col].value_counts().sort_values()
    fig, ax = plt.subplots(figsize=(10, max(4, len(counts)*0.35)))
    counts.plot(kind="barh", ax=ax)
    ax.set(title=f"Record count by {col}", xlabel="Records", ylabel=col)
    fig.tight_layout(); fig.savefig(f"{OUT}/bar_{col}.png", dpi=150); plt.close(fig)
    order = sub[col].value_counts().head(15).index.tolist()
    fig, ax = plt.subplots(figsize=(max(10, len(order)*0.65), 6))
    sns.boxplot(data=sub, x=col, y="ResultValue", order=order, showfliers=False, ax=ax)
    ax.set(title=f"ResultValue by {col} — boxplot", ylabel="ResultValue (original units)")
    ax.tick_params(axis="x", rotation=45); fig.tight_layout()
    fig.savefig(f"{OUT}/boxplot_{col}.png", dpi=150); plt.close(fig)
    sample = sub.sample(min(len(sub), 12000), random_state=42)
    fig, ax = plt.subplots(figsize=(max(10, len(order)*0.65), 6))
    sns.violinplot(data=sample, x=col, y="ResultValue", order=order, cut=0, inner="quartile", ax=ax)
    ax.set(title=f"ResultValue by {col} — violin plot", ylabel="ResultValue (original units)")
    ax.tick_params(axis="x", rotation=45); fig.tight_layout()
    fig.savefig(f"{OUT}/violin_{col}.png", dpi=150); plt.close(fig)
    for cat, grp in sub.groupby(col):
        summary.append({"category_column": col, "category": cat, "record_count": len(grp),
                        "median_ResultValue": grp["ResultValue"].median()})
pd.DataFrame(summary).to_csv(f"{OUT}/category_summary.csv", index=False)
print("Saved bar charts, boxplots, violin plots and category_summary.csv in", OUT)
print("For meaningful comparisons, filter to one CharacteristicName and compatible ResultUnit.")


Saved bar charts, boxplots, violin plots and category_summary.csv in task1_categorical_outputs
For meaningful comparisons, filter to one CharacteristicName and compatible ResultUnit.
